
# dGen Results — streamlined analysis

This notebook demonstrates a clean, single‑pass workflow built around
`DataWarehouse` and the revised coincident‑peak metrics.

> **Tip:** Edit `ROOT_DIR` and `RUN_ID` to point at your run directory.


In [ ]:
import sys, os
sys.path.append(os.path.abspath("../"))

In [ ]:

from pathlib import Path
import importlib

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns

import analysis_functions as af
importlib.reload(af)  # safe reload during editing

sns.set_context("talk")
sns.set_style("whitegrid")

ROOT_DIR = "/Users/wael/Documents/dgen_runs"      
RUN_ID   = "run_nc_single_state" 
STATES   = ["NC"]              
cfg = af.SavingsConfig(lifetime_years=25, cap_to_horizon=False)
#states = gpd.read_file("../../../data/states.shp")


## 1) Load once

In [ ]:
wh = af.DataWarehouse.from_disk(ROOT_DIR, run_id=RUN_ID, states=STATES)

## 2) Aggregations and peaks

In [ ]:
outputs = af.aggregate_state_metrics(wh.agents, cfg)
peaks   = af.compute_peaks_by_state(wh.state_hourly)

## 3) Coincident peak reduction — *top‑10 baseline hours* (recommended)

In [ ]:

rto_co_top10 = af.compute_rto_coincident_reduction(
    wh, method="baseline_topN_avg", top_n=10, return_by_rto=False
)

## 4) National lines (with coincident delta panel)

In [ ]:

af.facet_lines_national_totals(
    outputs=outputs,
    peaks_df=peaks,
    coincident_df=rto_co_top10,
    title="U.S. Totals — BAU vs Policy + Coincident Δ (top-10)",
    ncols=3,
)


## 5) Sensitivity: alternative coincident metrics

In [ ]:
# co_single   = af.compute_rto_coincident_reduction(wh, method="single_hour")
# co_sep_topN = af.compute_rto_coincident_reduction(wh, method="separate_topN_means", top_n=10)

In [ ]:
# af.facet_choropleth_payback_continuous(warehouse=wh, shapefile_path="../../../data/states.shp", year=2040, agents=wh.agents)

In [ ]:
af.plot_us_cum_adopters_grouped(outputs)

In [ ]:
af.bar_us_net_savings_median_2040_from_agents(
    warehouse=wh, agents = wh.agents, title=""
)

In [ ]:
af.plot_us_payback_customers_weighted_over_time(
    warehouse = wh, agents = wh.agents
)

In [ ]:
# if STATES is None:
#     af.choropleth_pct_households_with_solar(outputs, "../../../data/states.shp", year=2040)
# else: 
#     pass

In [ ]:
# if STATES is None:
#     af.choropleth_state_coincident_reduction(
#         warehouse=wh, shapefile_path="../../../data/states.shp", 
#         year=2040, method="baseline_topN_avg", top_n=10)
# else: 
#     pass

In [ ]:
eabs_states = af.build_eabs_calendar_timeseries(warehouse=wh)
eabs_us = af.build_eabs_calendar_timeseries(warehouse=wh, level="us")
eabs_states

In [ ]:
annual, cumulative, lifetime = af.compute_portfolio_and_cumulative_savings(wh.agents, cfg)

nat_ts = af.build_us_cumulative_timeseries(annual, lifetime_df=lifetime, xticks_full=True)

# e.g., get 2040 numbers
nat_2040 = af.summarize_us_cumulative_for_year(nat_ts, 2040)
print(nat_2040)

## 6) Optional: export tidy tables

In [ ]:
out_path = af.export_compiled_results_to_excel(outputs, run_id=RUN_ID or "run", out_dir=ROOT_DIR,
                                              peaks_df=peaks, coincident_df=rto_co_top10)

In [ ]:
# --- Diagnostic: year-by-year bill savings and cf_energy_value for a single agent ---
import ast

sample = wh.agents[wh.agents['scenario'] == 'policy'].iloc[0]

def parse_arr(v):
    if isinstance(v, list): return v
    try:
        s = str(v).strip()
        if s.startswith('{'): return [float(x) for x in s.strip('{}').split(',') if x.strip()]
        return list(ast.literal_eval(s))
    except Exception: return []

bill_wo = parse_arr(sample['utility_bill_wo_sys_pv_only'])
bill_w  = parse_arr(sample['utility_bill_w_sys_pv_only'])
cf_ev   = parse_arr(sample['cf_energy_value_pv_only'])

print(f"{'Year':<6} {'bill_wo':>10} {'bill_w':>10} {'savings':>10} {'cf_ev':>10}")
print('-' * 50)
for k in range(min(25, len(bill_wo), len(bill_w), len(cf_ev))):
    savings = bill_wo[k] - bill_w[k]
    print(f"{k:<6} {bill_wo[k]:>10,.0f} {bill_w[k]:>10,.0f} {savings:>10,.0f} {cf_ev[k]:>10,.0f}")
print()
print(f"25yr total bill savings: ${sum(bill_wo[k] - bill_w[k] for k in range(min(25, len(bill_wo), len(bill_w)))):,.0f}")
print(f"25yr total cf_ev:        ${sum(cf_ev[:25]):,.0f}")


In [ ]:
import pandas as pd
pd.read_sql("SELECT * FROM diffusion_shared.input_main_market_inflation", con)